# M2 - Preparacion de datos
Objetivo: construir df_clean

In [ ]:
import sys
import os
import pandas as pd
import pathlib import Path

# Agregar la raiz del proyecto al path para importar
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# Ruta al excel
DATA_PATH = os.path.join(ROOT, "data/raw", "/Users/robertosanchezsantoyo/Library/Mobile Documents/com~apple~CloudDocs/00_Main/03_Academic/Research/Investigacion_verano_2026/data/raw/2025 CPH.xlsx")

# Paso 1 - Carga y validacion inicial

In [ ]:
from src.m2_data_prep.loader import cargar_hoja1

df_raw = cargar_hoja1(DATA_PATH, verbose = True)

In [ ]:
# Vista general
df_raw.head(3)

In [ ]:
# Shape y dtypes - todo debe de ser object
print("Shape:", df_raw.shape)
print("\nDtypes únicos:", df_raw.dtypes.value_counts().to_dict())

In [ ]:
# Lista completa de columnas con su índice
for i, col in enumerate(df_raw.columns):
    print(f"{i:>3}  {col}")

In [ ]:
# Confirmar que el target existe
TARGET = "Processed WB (liters)"
assert TARGET in df_raw.columns, f"Columna target '{TARGET}' no encontrada"
print(f"Target '{TARGET}' presente en columna {df_raw.columns.tolist().index(TARGET)}")
print(df_raw[TARGET].head(10).tolist())

# Paso 2 - Parseo numerico

In [ ]:
from src.m2_data_prep.cleaning import parsear_numericos

df_parsed = parsear_numericos(df_raw, verbose = True)

In [ ]:
# Verificar dtypes: deben aparecer float, int y str
print("Shape: ", df_parsed.shape)
print("Dtypes: ", df_parsed.dtypes.value_counts().to_dict())

In [ ]:
# Inspeccion del target
print(df_parsed[TARGET].describe())

In [ ]:
from src.m2_data_prep.leakage import quitar_leakage

X, y = quitar_leakage(df_parsed, verbose = True)

In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("\ny describe:")
print(y.describe())

# Paso 4 - Encoding de categorias

In [ ]:
from src.m2_data_prep.encoding import encodear_categorias

X_enc = encodear_categorias(X, verbose=True)

In [ ]:
# Lista completa de features finales
for i, col in enumerate(X_enc.columns):
        print(f"{i:>2}  {col}")

In [ ]:
# Verificar NaN por columna en X_enc
nan_counts = X_enc.isna().sum()
nan_counts = nan_counts[nan_counts > 0].sort_values(ascending=False)
print("NaN por columna:")
print(nan_counts)

# Paso 5 - Persistir df_clean
Combinamos X_enc + y en un unico DataFrame y lo guardamos como parquete

In [ ]:
# Combinar X_enc + y
df_clean = X_enc.copy()
df_clean["Processed WB (liters)"] = y.values

# Guardar
out_path = Path(ROOT) / "data" / "processed" / "df_clean.parquet"
out_path.parent.mkdir(parents = True, exist_ok = True)
df_clean.to_parquet(out_path, index = False)

print(f"df_clean guardado en: {out_path}")
print(f"Shape: {df_clean.shape}")
print(f"Tamaño: {out_path.stat().st_size / 1024:.1f} KB")

In [ ]:
# Verificar leyendo el parquet desde cero
df_check = pd.read_parquet(out_path)
print("Shape:", df_check.shape)
print("NaN en y:", df_check["Processed WB (liters)"].isna().sum())
df_check.head(3)